
# vLLM Source-Level Project

**Today's goal:** Environment → Run → Serve → Measure → Locate source.

You are **not** optimizing vLLM today. By the end of the notebook you should be able to:

1. Confirm the Colab T4 environment.
2. Install and record the exact vLLM/PyTorch/CUDA environment.
3. Run offline inference through `vllm.LLM`.
4. Inspect `RequestOutput`.
5. Run a very small latency experiment.
6. Start the OpenAI-compatible vLLM server.
7. Send one request through the HTTP serving path.
8. Record observations for Day 2.

### Rules
- Cells marked **TODO — YOU WRITE THIS** are intentionally incomplete.
- Try not to look up the final implementation until you have attempted it.
- Do not optimize anything yet.
- Use a **T4 runtime**: Runtime → Change runtime type → T4 GPU.


## 0. Confirm the GPU

In [1]:

!nvidia-smi


Sat Sep 19 18:37:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----


### Checkpoint 0
Before continuing, answer in your notes:

- GPU model:
- Total VRAM:
- Driver version:
- CUDA version reported by `nvidia-smi`:


## 1. Install vLLM and helper packages

In [2]:
!pip uninstall -y torchaudio
!pip install -U torchaudio==2.11.0+cu130 \
  --index-url https://download.pytorch.org/whl/cu130

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu130
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.2 MB/s eta 0:00:00


In [3]:
# vLLM changes quickly. For Day 1 we use the released package and record
# the exact installed version below so future benchmarks are reproducible.
%pip install -q -U vllm openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2

In [ ]:
import os
os.kill(os.getpid(), 9)


> If Colab asks you to restart the runtime after installation, restart it and then rerun from Cell 0.


## 2. Record the experiment environment

In [1]:
import sys
import torch
import vllm
import time
import statistics

print("Python :", sys.version)
print("PyTorch:", torch.__version__)
print("Torch CUDA build:", torch.version.cuda)
print("vLLM   :", vllm.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print(f"VRAM: {props.total_memory / 1024**3:.2f} GiB")


Python : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.13.0+cu130
Torch CUDA build: 13.0
vLLM   : 0.29.0
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GiB


## 3. Offline inference — first core exercise

In [2]:
import torch
import torchaudio

print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)
print("torchaudio:", torchaudio.__version__)

torch: 2.13.0+cu130
torch CUDA: 13.0
torchaudio: 2.11.0+cu130


In [3]:
%%writefile /content/test_vllm.py
from vllm import LLM, SamplingParams
import time
import statistics

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.70

output_path = "/content/request_output_inspection.txt"

prompts = [
    "Explain in one sentence what a KV cache is in LLM inference.",
    "Why can continuous batching improve GPU utilization?",
]

# ============================================================
# TODO — YOU WRITE THIS
#
# Task A: Construct SamplingParams.
# Requirements:
#   - temperature = 0.0
#   - max_tokens = 64
#
# sampling_params = ...
# ============================================================

sampling_params = SamplingParams(temperature=0.0,
                                 max_tokens=64)

# ============================================================
# TODO — YOU WRITE THIS
#
# Task B: Construct the vLLM LLM object.
#
# Requirements:
#   model=MODEL
#   dtype="float16"
#   max_model_len=MAX_MODEL_LEN
#   gpu_memory_utilization=GPU_MEMORY_UTILIZATION
#
# llm = ...
# ============================================================

llm = LLM(model = MODEL,
          dtype = "float16",
          max_model_len = MAX_MODEL_LEN,
          gpu_memory_utilization = GPU_MEMORY_UTILIZATION)

# ============================================================
# TODO — YOU WRITE THIS
#
# Task C: Generate outputs for `prompts`.
#
# Ask yourself:
#   1. Which object owns the generate() method?
#   2. What two main arguments does it need here?
#
# outputs = ...
# ============================================================

outputs = llm.generate(
    prompts,
    sampling_params
)


# ============================================================
# Inspect RequestOutput
# Small offline timing experiment:
# ============================================================

short_prompt = "Briefly explain GPU memory bandwidth."
long_prompt = ("Explain the relationship between GPU compute throughput, memory "
               "bandwidth, arithmetic intensity, and kernel performance. " * 80)

def run_once(prompt):
    # ========================================================
    # TODO — YOU WRITE THIS
    #
    # Measure wall-clock latency around ONE llm.generate call.
    # Return:
    #   latency_seconds, output_token_count
    #
    # Notes:
    # - This is NOT yet a rigorous serving benchmark.
    # - llm.generate is offline/batched inference, not HTTP TTFT.
    # - We only want a Day-1 baseline and familiarity with outputs.
    # ========================================================
    # raise NotImplementedError
    start_time = time.perf_counter()
    output = llm.generate(
        [prompt],
        sampling_params
    )
    end_time = time.perf_counter()
    latency_seconds = end_time - start_time
    output_token_count = len(output[0].outputs[0].token_ids)
    return latency_seconds, output_token_count

# Warm-up: complete this after run_once() works.
# _ = run_once(short_prompt)

# ============================================================
# TODO — YOU WRITE THIS
#
# Run each prompt 3 times and report:
#   mean latency
#   mean output-token count
#
# Compare short_prompt vs long_prompt.
# ============================================================
short_results = [run_once(short_prompt) for _ in range(3)]
long_results = [run_once(long_prompt) for _ in range(3)]

short_latencies = [x[0] for x in short_results]
short_tokens = [x[1] for x in short_results]

long_latencies = [x[0] for x in long_results]
long_tokens = [x[1] for x in long_results]

with open(output_path, "w", encoding="utf-8") as f:

    # ========================================================
    # RequestOutput inspection
    # ========================================================
    f.write("=== RequestOutput Inspection ===\n\n")

    for output in outputs:
        candidate = output.outputs[0]

        f.write(f"Request ID: {output.request_id}\n")
        f.write(f"Prompt: {output.prompt}\n")
        f.write(f"Prompt token count: {len(output.prompt_token_ids)}\n")
        f.write(f"Finished: {output.finished}\n")

        f.write(f"Generated text: {candidate.text}\n")
        f.write(f"Generated token IDs: {candidate.token_ids}\n")
        f.write(f"Generated token count: {len(candidate.token_ids)}\n")
        f.write(f"Finish reason: {candidate.finish_reason}\n")

        f.write("-" * 60 + "\n")

    # ========================================================
    # Latency benchmark
    # ========================================================
    f.write("\n=== Offline Latency Benchmark ===\n\n")

    f.write("Short prompt:\n")
    f.write(f"Mean latency: {statistics.mean(short_latencies):.4f} s\n")
    f.write(f"Mean output tokens: {statistics.mean(short_tokens):.2f}\n")
    f.write(f"Raw latencies: {short_latencies}\n")
    f.write("\n")

    f.write("Long prompt:\n")
    f.write(f"Mean latency: {statistics.mean(long_latencies):.4f} s\n")
    f.write(f"Mean output tokens: {statistics.mean(long_tokens):.2f}\n")
    f.write(f"Raw latencies: {long_latencies}\n")

Writing /content/test_vllm.py


In [4]:
!python /content/test_vllm.py

INFO 09-19 18:43:07 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
config.json: 100% 660/660 [00:00<00:00, 3.83MB/s]
INFO 09-19 18:43:25 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-19 18:43:25 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-19 18:43:25 [model.py:2021] Using max model len 4096
INFO 09-19 18:43:25 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-19 18:43:26 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 21.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 74.7MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 121MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 141MB/s]
generation_config.json: 100% 242/242 [00:00<00:00,


### What you should notice

The important conceptual call today is:

```text
Python program
    ↓
LLM.generate(...)
    ↓
vLLM engine
    ↓
scheduler / KV cache / model runner
    ↓
GPU
    ↓
RequestOutput
```

Today we treat the middle as a black box. Day 2 starts opening it.


## 4. Inspect `RequestOutput`

## 5. Small offline timing experiment:


### Checkpoint 1 — write down what happened

Do **not** over-interpret the numbers yet.

| Workload | Mean latency | Mean generated tokens |
|---|---:|---:|
| short prompt | | |
| long prompt | | |

Questions:

1. Did the longer prompt increase latency?
2. Why might that happen?
3. Is this number TTFT? Why or why not?
4. Which stage should be more affected by prompt length: prefill or decode?


## 6. Inspect GPU memory after model load

In [5]:

print(torch.cuda.memory_summary(abbreviated=True))
!nvidia-smi


|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Requested memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------


### Checkpoint 2

Record:

- GPU memory used after model load:
- Approximate free memory:
- Why does vLLM reserve significant GPU memory beyond model weights?
- What future structure do you expect to occupy much of that memory?

Do not worry if you cannot fully explain the last two yet — that is Day 4.



## 7. Release the offline engine before starting the server

A single T4 cannot comfortably host two copies of the same vLLM model.
Delete the offline engine and clear Python's references before starting a separate server process.


In [6]:

import gc

# Keep this infrastructure code as-is.
# del outputs
# del llm
gc.collect()
torch.cuda.empty_cache()

!nvidia-smi


Sat Sep 19 18:45:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P0             28W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----


## 8. Start the OpenAI-compatible vLLM server

The server uses the same model but exposes it over HTTP.

This cell is infrastructure, so it is provided for you.

If it fails because a CLI flag changed in your installed vLLM version, inspect:

```bash
vllm serve --help
```


In [7]:

import subprocess
import time
import os
import signal

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.70

server_log = open("/content/vllm_server.log", "w")

server = subprocess.Popen(
    [
        "vllm", "serve", MODEL,
        "--dtype", "float16",
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--host", "127.0.0.1",
        "--port", "8000",
    ],
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

print("Server PID:", server.pid)
print("Log: /content/vllm_server.log")


Server PID: 11106
Log: /content/vllm_server.log


## 9. Wait until the server is healthy

In [8]:

import requests
import time

for i in range(60):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            print("Server is ready.")
            break
    except Exception:
        pass

    if i % 5 == 0:
        print(f"Waiting... ({i})")
    time.sleep(2)
else:
    print("Server did not become healthy. Inspect the log:")
    print(open("/content/vllm_server.log").read()[-8000:])


Waiting... (0)
Waiting... (5)
Waiting... (10)
Waiting... (15)
Waiting... (20)
Waiting... (25)
Waiting... (30)
Waiting... (35)
Waiting... (40)
Waiting... (45)
Waiting... (50)
Waiting... (55)
Server is ready.


## 10. Send your first HTTP request — second core exercise

In [9]:

from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:8000/v1",
    api_key="EMPTY",
)

# ============================================================
# TODO — YOU WRITE THIS
#
# Send ONE chat completion request.
#
# Requirements:
#   model=MODEL
#   one user message asking:
#       "What problem does PagedAttention solve?"
#   temperature=0.0
#   max_tokens=64
#
# response = client.chat.completions.create(...)
# ============================================================

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "What problem does PagedAttention solve?"
        }
    ],
    temperature=0.0,
    max_tokens=64
)

# ============================================================
# TODO — YOU WRITE THIS
#
# Print only the generated assistant text.
# Explore `response` if you don't know the object layout.
# ============================================================
print(response.choices[0].message.content)

I'm sorry, but I couldn't find any information about a specific problem called "PagedAttention" that needs to be solved. It's possible that there might be some confusion or typo in the name.

Could you please provide more details or context about what kind of problem you're referring to? This will help me



## 11. Observe streaming and approximate TTFT

This is the first place today where you should distinguish:

- **TTFT**: request sent → first generated token/chunk arrives
- **E2E latency**: request sent → generation finishes

For Day 1, measuring Python streaming-chunk arrival time is sufficient.
It is not yet our final benchmark methodology.


In [10]:

import time

# ============================================================
# TODO — YOU WRITE THIS
#
# Send a streaming Chat Completions request.
#
# Measure:
#   start_time
#   first_nonempty_content_time
#   finish_time
#
# Compute:
#   approximate_TTFT = first_nonempty_content_time - start_time
#   E2E_latency      = finish_time - start_time
#
# Requirements:
#   prompt: "Explain continuous batching in about 100 words."
#   temperature=0.0
#   max_tokens=128
#   stream=True
#
# Important:
# Some initial stream chunks may contain metadata/role with no text.
# Count TTFT at the FIRST NON-EMPTY generated content chunk.
# ============================================================
start_time = time.perf_counter()
first_nonempty_content_time = None

stream = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Explain continuous batching in about 100 words."
        }
    ],
    temperature=0.0,
    max_tokens=128,
    stream=True
)

generated_text = ""

for chunk in stream:
    content = chunk.choices[0].delta.content

    if content:
        if first_nonempty_content_time is None:
            first_nonempty_content_time = time.perf_counter()

        generated_text += content
        print(content, end="", flush=True)

finish_time = time.perf_counter()

approximate_TTFT = first_nonempty_content_time - start_time
E2E_latency = finish_time - start_time

print("\n")
print(f"Approximate TTFT: {approximate_TTFT:.4f} s")
print(f"E2E latency:      {E2E_latency:.4f} s")

Continuous Batching is an optimization technique used in machine learning and data processing pipelines to improve efficiency and reduce latency. It involves dividing the input data into smaller batches that can be processed concurrently without waiting for previous batches to complete.

In traditional batch processing, all data is loaded into memory at once before being processed. This approach works well when dealing with small datasets or when there's sufficient computational resources available. However, it becomes inefficient as the size of the dataset grows due to increased memory usage and slower performance bottlenecks.

Continuous Batching addresses these issues by breaking down large datasets into multiple smaller batches during each iteration of training or processing. Each batch

Approximate TTFT: 0.0763 s
E2E latency:      1.9103 s



### Checkpoint 3

Record:

- Approximate TTFT:
- End-to-end latency:
- Generated output tokens (if you counted them):
- Why is TTFT different from total latency?
- Which part of inference dominates TTFT for a long prompt?


## 12. Locate the installed vLLM source tree

In [11]:

import inspect
import os
import vllm

vllm_root = os.path.dirname(inspect.getfile(vllm))
print("Installed vLLM source:", vllm_root)

# We only LOCATE files today. Do not dive deeply into them yet.
!find "$vllm_root/v1" -maxdepth 3 -type f | grep -E "(engine|scheduler|kv_cache|model_runner)" | head -80


Installed vLLM source: /usr/local/lib/python3.13/dist-packages/vllm
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/exceptions.py
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/detokenizer.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/utils.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/exceptions.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/__init__.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/core.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/parallel_sampling.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/output_processor.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/core_client.cpython-313.pyc
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/__pycache__/async_llm.cpython-313.pyc
/usr/local/lib/p


## 13. Day-1 source map — fill this yourself

Without trying to understand every line, locate likely files/classes for:

```text
User / API
    ↓
LLM / serving frontend
    ↓
Engine / EngineCore
    ↓
Scheduler
    ↓
KV-cache management
    ↓
Model runner
    ↓
GPU
```

Fill in:

| Layer | File / class you found | What you THINK it does |
|---|---|---|
| Offline frontend | | |
| Engine / core | | |
| Scheduler | | |
| KV cache | | |
| Model runner | | |

It is okay if some guesses are wrong. Day 2 will verify the request path.


## 14. Stop the server

In [12]:

# Infrastructure cleanup.
if server.poll() is None:
    server.terminate()
    try:
        server.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server.kill()

server_log.close()
print("Server stopped.")


Server stopped.


# Day 2 — Tracing the vLLM Request Lifecycle

## Objective

The goal of Day 2 is to open the `LLM.generate()` black box and understand how a user prompt becomes an internal vLLM request before being scheduled for GPU execution.

Day 1 established that inference works correctly through vLLM. Day 2 focuses on the control flow behind that API call.

The main path to trace is:

```text
User Prompt
    ↓
LLM.generate()
    ↓
Internal completion path
    ↓
Request creation / preprocessing
    ↓
Engine / EngineCore
    ↓
Scheduler.add_request()
    ↓
Waiting queue
    ↓
Scheduler.schedule()
```

The goal is not to understand every line of vLLM source code. Instead, the focus is to identify the main objects, functions, and state transitions that a request passes through.

---

## What We Want to Understand

By the end of Day 2, we should be able to answer the following questions:

1. Where is `LLM.generate()` implemented?
2. Which internal function does `generate()` call next?
3. How is a user prompt converted into an internal request?
4. What information is stored inside the request object?
5. How does the request reach `EngineCore`?
6. How is the request inserted into the scheduler?
7. When does the request enter the waiting queue?
8. What does `Scheduler.schedule()` produce for the next execution step?

---

## Mental Model

At the beginning of Day 2, the system still looks like this:

```text
llm.generate(...)
      ↓
   black box
      ↓
RequestOutput
```

The purpose of Day 2 is to expand the black box into:

```text
llm.generate(...)
      ↓
LLM.generate()
      ↓
internal completion path
      ↓
request preprocessing
      ↓
internal request object
      ↓
EngineCore
      ↓
Scheduler.add_request()
      ↓
waiting queue
      ↓
Scheduler.schedule()
      ↓
scheduled workload
```

This request lifecycle will become the foundation for understanding continuous batching and scheduler behavior in later stages of the project.

---

## Source-Level Goals

During Day 2, we will use Python introspection tools such as:

```python
inspect.getfile(...)
inspect.getsource(...)
```

to locate and inspect the vLLM implementation currently installed in the environment.

The purpose is to follow the actual installed vLLM version rather than relying only on documentation or diagrams.

We will progressively trace:

```text
LLM.generate()
→ internal helper functions
→ request creation
→ engine submission
→ scheduler admission
```

At each step, we will record:

* source file
* class or function name
* input object
* output object
* role in the request lifecycle

---

## Request Lifecycle Table

As we trace the source code, we will gradually fill in the following table:

| Stage                 | File / Class | Function         | Purpose                                                              |
| --------------------- | ------------ | ---------------- | -------------------------------------------------------------------- |
| Public API            |              | `LLM.generate()` | Entry point for offline inference                                    |
| Completion path       |              |                  | Converts public API call into internal processing                    |
| Request preprocessing |              |                  | Converts prompt and sampling parameters into internal representation |
| Engine                |              |                  | Submits request to execution core                                    |
| EngineCore            |              |                  | Coordinates scheduling and model execution                           |
| Scheduler admission   |              | `add_request()`  | Inserts request into scheduler                                       |
| Waiting queue         |              |                  | Holds requests not currently executing                               |
| Scheduling            |              | `schedule()`     | Selects work for the next engine step                                |

---

## What We Are NOT Doing Today

Day 2 is about request flow, not scheduler optimization.

We will not yet deeply analyze:

* token budgets
* `num_computed_tokens`
* continuous batching policy
* chunked prefill policy
* KV-cache block allocation
* preemption
* scheduler optimization

Those topics will be handled in later stages.

The important goal today is simply:

> Understand how a prompt becomes a scheduler-visible request.

---

## Expected Output

At the end of Day 2, we should have a request lifecycle diagram similar to:

```text
User Prompt
    ↓
LLM.generate()
    ↓
Request preprocessing
    ↓
Internal Request
    ↓
EngineCore
    ↓
Scheduler.add_request()
    ↓
WAITING
    ↓
Scheduler.schedule()
    ↓
RUNNING / Scheduled Work
```

We should also be able to explain this process without looking at the source code.

---

## Day 2 Summary Goal

**Trace a request from `LLM.generate()` into the vLLM engine and scheduler, identifying the main source files, internal request representation, and queue transitions involved in request admission.**


In [13]:
"""
Day 2 — Tracing the vLLM Request Lifecycle

Goal:
    Trace the control flow from LLM.generate() toward the internal
    request-processing path and scheduler entry points.

This script focuses on source inspection rather than performance.
"""

import inspect

from vllm import LLM


def print_section(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


# =============================================================================
# Step 1 — Locate LLM.generate()
# =============================================================================

print_section("Step 1 — Locate LLM.generate()")

# TODO:
# Use inspect.getfile(...) to find the source file that contains LLM.generate().
#
# Expected concept:
#     public API method
#         ↓
#     actual Python source file
#
# Hint:
#     inspect.getfile(...)
#
generate_file = inspect.getfile(LLM.generate)

print("LLM.generate() source file:")
print(generate_file)


Step 1 — Locate LLM.generate()
LLM.generate() source file:
/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/llm.py


In [14]:
# =============================================================================
# Step 2 — Inspect the source code of LLM.generate()
# =============================================================================

print_section("Step 2 — Inspect LLM.generate() source")

# TODO:
# Use inspect.getsource(...) to retrieve the implementation of LLM.generate().
#
# After printing the source, identify:
#   1. What function does generate() eventually call?
#   2. Which arguments are forwarded?
#
generate_source = inspect.getsource(LLM.generate)

print(generate_source)


Step 2 — Inspect LLM.generate() source
    def generate(
        self,
        prompts: PromptType | Sequence[PromptType],
        sampling_params: SamplingParams | Sequence[SamplingParams] | None = None,
        *,
        use_tqdm: bool | Callable[..., tqdm] = True,
        lora_request: Sequence[LoRARequest] | LoRARequest | None = None,
        priority: list[int] | None = None,
        tokenization_kwargs: dict[str, Any] | None = None,
        mm_processor_kwargs: dict[str, Any] | None = None,
    ) -> list[RequestOutput]:
        """Generates the completions for the input prompts.

        This class automatically batches the given prompts, considering
        the memory constraint. For the best performance, put all of your prompts
        into a single list and pass it to this method.

        Args:
            prompts: The prompts to the LLM. You may pass a sequence of prompts
                for batch inference. See [PromptType][vllm.inputs.PromptType]
                for more

In [15]:
# =============================================================================
# Step 3 — Find the next internal method
# =============================================================================

print_section("Step 3 — Inspect the next internal completion method")

# TODO:
# Based on the source code from Step 2, identify the next important
# internal method called by generate().
#
# Example reasoning:
#
#     LLM.generate()
#         ↓
#     some_internal_method(...)
#
# Replace the placeholder below with the correct bound method.
#
next_method = LLM._run_completion

# TODO:
# Print the source file of the next method.
#
next_method_file = inspect.getfile(next_method)

print("Next method source file:")
print(next_method_file)


# TODO:
# Print the source code of the next method.
#
next_method_source = inspect.getsource(next_method)

print(next_method_source)


Step 3 — Inspect the next internal completion method
Next method source file:
/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/offline_utils.py
    def _run_completion(
        self,
        prompts: PromptType | Sequence[PromptType],
        params: SamplingParams
        | PoolingParams
        | Sequence[SamplingParams | PoolingParams],
        output_type: type[_O],
        *,
        use_tqdm: bool | Callable[..., tqdm] = True,
        lora_request: Sequence[LoRARequest] | LoRARequest | None = None,
        priority: list[int] | None = None,
        tokenization_kwargs: dict[str, Any] | None = None,
        mm_processor_kwargs: dict[str, Any] | None = None,
    ):
        self._add_completion_requests(
            prompts=prompts,
            params=params,
            use_tqdm=use_tqdm,
            lora_request=lora_request,
            priority=priority,
            tokenization_kwargs=tokenization_kwargs,
            mm_processor_kwargs=mm_processor_kwargs,
        )
 

In [16]:
# =============================================================================
# Step 4 — Inspect request submission
# =============================================================================

print_section("Step 4 — Inspect _add_completion_requests()")

# TODO:
# Locate the internal method responsible for converting prompts
# into requests and submitting them into the engine.
#
# Hint:
# The method name was discovered in Step 3.
#
request_submission_method = LLM._add_completion_requests

# TODO:
# Find the source file.
request_submission_file = inspect.getfile(request_submission_method)

print("Request submission method source file:")
print(request_submission_file)


# TODO:
# Print the source code.
request_submission_source = inspect.getsource(request_submission_method)

print(request_submission_source)


Step 4 — Inspect _add_completion_requests()
Request submission method source file:
/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/offline_utils.py
    def _add_completion_requests(
        self,
        prompts: PromptType | Sequence[PromptType],
        params: SamplingParams
        | PoolingParams
        | Sequence[SamplingParams | PoolingParams],
        *,
        use_tqdm: bool | Callable[..., tqdm] = True,
        lora_request: Sequence[LoRARequest] | LoRARequest | None = None,
        priority: list[int] | None = None,
        tokenization_kwargs: dict[str, Any] | None = None,
        mm_processor_kwargs: dict[str, Any] | None = None,
    ) -> list[str]:
        seq_prompts = prompt_to_seq(prompts)
        seq_params = self._params_to_seq(params, len(seq_prompts))
        seq_lora_requests = self._lora_request_to_seq(lora_request, len(seq_prompts))
        seq_priority = self._priority_to_seq(priority, len(seq_prompts))

        return self._render_and_add_requests(

In [17]:
# =============================================================================
# Step 5 — Inspect _render_and_add_requests()
# =============================================================================

print_section("Step 5 — Inspect _render_and_add_requests()")

# TODO:
# Locate the method that receives preprocessed prompts and adds requests
# into the engine.
#
# Hint:
# It was called at the end of _add_completion_requests().
#
render_add_method = LLM._render_and_add_requests

# TODO:
# Find its source file.
render_add_file = inspect.getfile(render_add_method)

print("Source file:")
print(render_add_file)

# TODO:
# Print the source code.
render_add_source = inspect.getsource(render_add_method)

print(render_add_source)


Step 5 — Inspect _render_and_add_requests()
Source file:
/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/offline_utils.py
    def _render_and_add_requests(
        self,
        prompts: Iterable[EngineInput],
        params: Sequence[SamplingParams | PoolingParams],
        *,
        lora_requests: Sequence[LoRARequest | None] | None = None,
        priorities: Sequence[int] | None = None,
    ) -> list[str]:
        added_request_ids: list[str] = []

        try:
            for i, prompt in enumerate(prompts):
                request_id = self._add_request(
                    prompt,
                    params[i],
                    lora_request=self._resolve_mm_lora(
                        prompt,
                        None if lora_requests is None else lora_requests[i],
                    ),
                    priority=0 if priorities is None else priorities[i],
                )
                added_request_ids.append(request_id)
        except Exception as e:


In [18]:
# =============================================================================
# Step 6 — Inspect _add_request()
# =============================================================================

print_section("Step 6 — Inspect _add_request()")

# TODO:
# Locate the method that submits one request deeper into the engine.
#
# Questions to answer:
#   1. Where is request_id created?
#   2. Which engine method receives the request?
#   3. What arguments are passed into the engine?
#
add_request_method = LLM._add_request

# TODO:
# Find the source file.
add_request_file = inspect.getfile(add_request_method)

print("Source file:")
print(add_request_file)

# TODO:
# Print the source code.
add_request_source = inspect.getsource(add_request_method)

print(add_request_source)


Step 6 — Inspect _add_request()
Source file:
/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/offline_utils.py
    def _add_request(
        self,
        prompt: EngineInput,
        params: SamplingParams | PoolingParams,
        lora_request: LoRARequest | None = None,
        priority: int = 0,
    ) -> str:
        if isinstance(params, SamplingParams):
            # We only care about the final output
            params.output_kind = RequestOutputKind.FINAL_ONLY

        request_id = str(next(self.request_counter))

        return self.llm_engine.add_request(
            request_id,
            prompt,
            params,
            lora_request=lora_request,
            priority=priority,
        )



In [19]:
# =============================================================================
# Step 7 — Inspect LLMEngine.add_request()
# =============================================================================

print_section("Step 7 — Inspect LLMEngine.add_request()")

# TODO:
# Identify the class of self.llm_engine.
#
# Questions:
#   1. What class owns add_request()?
#   2. Where is add_request() implemented?
#   3. What transformation happens before the request reaches EngineCore?
#

from vllm.engine.llm_engine import LLMEngine

engine_add_method = LLMEngine.add_request

# TODO:
# Locate the source file.
engine_add_file = inspect.getfile(engine_add_method)

print("Source file:")
print(engine_add_file)

# TODO:
# Print the source code.
engine_add_source = inspect.getsource(engine_add_method)

print(engine_add_source)


Step 7 — Inspect LLMEngine.add_request()
Source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/llm_engine.py
    def add_request(
        self,
        request_id: str,
        prompt: EngineCoreRequest | PromptType | EngineInput,
        params: SamplingParams | PoolingParams,
        arrival_time: float | None = None,
        lora_request: LoRARequest | None = None,
        tokenization_kwargs: dict[str, Any] | None = None,
        trace_headers: Mapping[str, str] | None = None,
        priority: int = 0,
        session_id: str | None = None,
        prompt_text: str | None = None,
    ) -> str:
        # Validate the request_id type.
        if not isinstance(request_id, str):
            raise TypeError(f"request_id must be a string, got {type(request_id)}")

        # Process raw inputs into the request.
        if isinstance(prompt, EngineCoreRequest):
            logger.warning_once(
                "Passing EngineCoreRequest to LLMEngine.generate() and .add_requ

In [20]:
# =============================================================================
# Step 8 — Inspect InputProcessor.process_inputs()
# =============================================================================

print_section("Step 8 — Inspect InputProcessor.process_inputs()")

# TODO:
# Identify the class that owns self.input_processor.
#
# Questions:
#   1. What does process_inputs() return?
#   2. Where is EngineCoreRequest created?
#   3. Which fields are copied into that request?
#
from vllm.v1.engine.input_processor import InputProcessor

input_processor = InputProcessor

# TODO:
# Find the class / method source file.
input_processor_method = InputProcessor.process_inputs
input_processor_file = inspect.getfile(input_processor_method)

print("InputProcessor.process_inputs() source file:")
print(input_processor_file)

# TODO:
# Print the source code.
input_processor_source = inspect.getsource(input_processor_method)


print(input_processor_source)


Step 8 — Inspect InputProcessor.process_inputs()
InputProcessor.process_inputs() source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/input_processor.py
    def process_inputs(
        self,
        request_id: str,
        prompt: PromptType | EngineInput,
        params: SamplingParams | PoolingParams,
        supported_tasks: tuple[SupportedTask, ...],
        arrival_time: float | None = None,
        lora_request: LoRARequest | None = None,
        tokenization_kwargs: dict[str, Any] | None = None,
        trace_headers: Mapping[str, str] | None = None,
        priority: int = 0,
        data_parallel_rank: int | None = None,
        resumable: bool = False,
        session_id: str | None = None,
    ) -> EngineCoreRequest:
        self._validate_params(params, supported_tasks)
        self._validate_lora(lora_request)

        parallel_config = self.vllm_config.parallel_config
        dp_size = parallel_config.data_parallel_size
        dp_local_size = parallel_co

In [21]:
# =============================================================================
# Step 9 — Trace EngineCoreRequest into the core client
# =============================================================================

print_section("Step 9 — Trace EngineCoreRequest into the core client")

from vllm.v1.engine.llm_engine import LLMEngine

# TODO:
# Inspect the complete implementation of LLMEngine.add_request().
#
# Questions:
#   1. After InputProcessor.process_inputs() returns EngineCoreRequest,
#      which object receives the request next?
#
#   2. Is the request also registered with an output processor?
#
#   3. Which method actually forwards the EngineCoreRequest toward EngineCore?
#
engine_add_method = LLMEngine.add_request

# TODO:
# Find the source file.
engine_add_file = inspect.getfile(engine_add_method)

print("LLMEngine.add_request() source file:")
print(engine_add_file)

# TODO:
# Retrieve and print the complete source code.
engine_add_source = inspect.getsource(engine_add_method)

print(engine_add_source)


Step 9 — Trace EngineCoreRequest into the core client
LLMEngine.add_request() source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/llm_engine.py
    def add_request(
        self,
        request_id: str,
        prompt: EngineCoreRequest | PromptType | EngineInput,
        params: SamplingParams | PoolingParams,
        arrival_time: float | None = None,
        lora_request: LoRARequest | None = None,
        tokenization_kwargs: dict[str, Any] | None = None,
        trace_headers: Mapping[str, str] | None = None,
        priority: int = 0,
        session_id: str | None = None,
        prompt_text: str | None = None,
    ) -> str:
        # Validate the request_id type.
        if not isinstance(request_id, str):
            raise TypeError(f"request_id must be a string, got {type(request_id)}")

        # Process raw inputs into the request.
        if isinstance(prompt, EngineCoreRequest):
            logger.warning_once(
                "Passing EngineCoreRequest 

In [22]:
# =============================================================================
# Step 10 — Inspect EngineCore.add_request()
# =============================================================================

print_section("Step 10 — Inspect EngineCore.add_request()")

# TODO:
# Identify the class behind self.engine_core and inspect add_request().
#
# Questions:
#   1. Does EngineCore directly own the Scheduler?
#   2. Is EngineCore.add_request() synchronous or forwarded through a client?
#   3. Where does Scheduler.add_request() get called?
#
from vllm.v1.engine.core import EngineCore
# engine_core_object = EngineCore

# TODO:
# Identify the class of the engine core object.
engine_core_class = EngineCore

print("EngineCore class:")
print(engine_core_class)

# TODO:
# Get the add_request() method from the class.
engine_core_add_method = EngineCore.add_request

# TODO:
# Find its source file.
engine_core_add_file = inspect.getfile(engine_core_add_method)

print("EngineCore.add_request() source file:")
print(engine_core_add_file)

# TODO:
# Print the source code.
engine_core_add_source = inspect.getsource(engine_core_add_method)

print(engine_core_add_source)


Step 10 — Inspect EngineCore.add_request()
EngineCore class:
<class 'vllm.v1.engine.core.EngineCore'>
EngineCore.add_request() source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/engine/core.py
    def add_request(self, request: Request, request_wave: int = 0):
        """Add request to the scheduler.

        `request_wave`: indicate which wave of requests this is expected to
        belong to in DP case
        """
        # Validate the request_id type.
        if not isinstance(request.request_id, str):
            raise TypeError(
                f"request_id must be a string, got {type(request.request_id)}"
            )

        if pooling_params := request.pooling_params:
            supported_pooling_tasks = [
                task for task in self.get_supported_tasks() if task in POOLING_TASKS
            ]

            if pooling_params.task not in supported_pooling_tasks:
                raise ValueError(
                    f"Unsupported task: {pooling_params.task

In [23]:
# =============================================================================
# Step 11 — Inspect Scheduler.add_request()
# =============================================================================

print_section("Step 11 — Inspect Scheduler.add_request()")

from vllm.v1.core.sched.scheduler import Scheduler

# TODO:
# Inspect Scheduler.add_request().
#
# Questions:
#   1. Which queue does a new request enter?
#   2. Is request priority considered here?
#   3. Does add_request() perform execution, or only admission?
#

scheduler_add_method = Scheduler.add_request

# TODO:
# Locate its source file.
scheduler_add_file = inspect.getfile(scheduler_add_method)

print("Scheduler.add_request() source file:")
print(scheduler_add_file)

# TODO:
# Print the source code.
scheduler_add_source = inspect.getsource(scheduler_add_method)

print(scheduler_add_source)


Step 11 — Inspect Scheduler.add_request()
Scheduler.add_request() source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/sched/scheduler.py
    def add_request(self, request: Request) -> None:
        existing = self.requests.get(request.request_id)
        if existing is not None:
            update = StreamingUpdate.from_request(request)
            if existing.status != RequestStatus.WAITING_FOR_STREAMING_REQ:
                assert existing.streaming_queue is not None, "duplicate request id"
                # Queue next input chunk (or finished sentinel).
                existing.streaming_queue.append(update)
            elif update is not None:
                # Commence next input chunk.
                self._update_request_as_session(existing, update)
            else:
                # Streaming-input session finished.
                self.finish_requests(request.request_id, RequestStatus.FINISHED_ABORTED)
        else:
            if request.resumable:
           

In [24]:
# =============================================================================
# Step 12 — Inspect Scheduler.schedule()
# =============================================================================

print_section("Step 12 — Inspect Scheduler.schedule()")

# TODO:
# Inspect Scheduler.schedule().
#
# Questions:
#   1. Does it examine waiting requests, running requests, or both?
#   2. How does it decide how many tokens to schedule?
#   3. What object does it return for the next engine step?
#
scheduler_schedule_method = Scheduler.schedule

# TODO:
# Locate the source file.
scheduler_schedule_file = inspect.getfile(scheduler_schedule_method)

print("Scheduler.schedule() source file:")
print(scheduler_schedule_file)

# TODO:
# Print the source code.
scheduler_schedule_source = inspect.getsource(scheduler_schedule_method)

print(scheduler_schedule_source)


Step 12 — Inspect Scheduler.schedule()
Scheduler.schedule() source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/sched/scheduler.py
    def schedule(self, throttle_prefills: bool = False) -> SchedulerOutput:
        self.current_step += 1
        # NOTE(woosuk) on the scheduling algorithm:
        # There's no "decoding phase" nor "prefill phase" in the scheduler.
        # Each request just has the num_computed_tokens and
        # num_tokens_with_spec. num_tokens_with_spec =
        # len(prompt_token_ids) + len(output_token_ids) + len(spec_token_ids).
        # At each step, the scheduler tries to assign tokens to the requests
        # so that each request's num_computed_tokens can catch up its
        # num_tokens_with_spec. This is general enough to cover
        # chunked prefills, prefix caching, speculative decoding,
        # and the "jump decoding" optimization in the future.

        scheduled_new_reqs: list[Request] = []
        scheduled_resumed_reqs: list[R

## Scheduler Flow

```text
Scheduler.schedule()
        ↓
Create token budget
        ↓
Schedule RUNNING requests
        ↓
Calculate num_new_tokens
        ↓
Allocate KV cache slots
        ↓
Schedule WAITING requests
        ↓
WAITING → RUNNING
        ↓
Record scheduled token counts
        ↓
Build SchedulerOutput
        ↓
ModelRunner / Execution

## Scheduler Execution Flow

```text
User Prompt
    ↓
LLM.generate()
    ↓
_run_completion()
    ↓
_add_completion_requests()
    ↓
_render_and_add_requests()
    ↓
_add_request()
    ↓
LLMEngine.add_request()
    ↓
InputProcessor.process_inputs()
    ↓
EngineCoreRequest
    ↓
EngineCore.add_request()
    ↓
Scheduler.add_request()
    ↓
WAITING
    ↓
Scheduler.schedule()
    ↓
Token-budget decision
    ↓
KV-cache allocation
    ↓
RUNNING
    ↓
SchedulerOutput
    ↓
Model Execution

# Day 3 — Understanding vLLM Scheduler Internals

## Objective

The goal of Day 3 is to understand how vLLM decides which requests run in each scheduler step and how many tokens each request is allowed to compute.

Day 2 traced the request lifecycle from:

```text
LLM.generate()
    ↓
LLMEngine
    ↓
EngineCore
    ↓
Scheduler.add_request()
    ↓
WAITING
```

Day 3 starts from `Scheduler.schedule()` and focuses on the scheduling policy itself.

The key question is:

> How does vLLM allocate limited per-step compute and KV-cache resources across multiple active requests?

---

## Main Concepts

Day 3 focuses on five core concepts:

### 1. Waiting and Running Requests

New requests first enter the scheduler's waiting queue.

```text
New Request
    ↓
WAITING
```

A request moves to the running queue only after the scheduler successfully allocates execution resources.

```text
WAITING
    ↓
token budget available
    ↓
KV-cache allocation succeeds
    ↓
RUNNING
```

Running requests remain eligible for future scheduler steps until generation finishes or they are preempted.

---

### 2. Token Budget

Each scheduler step has a limited token budget.

Conceptually:

```text
Scheduler Step
    ↓
Available Token Budget
    ↓
Distributed Across Requests
```

The scheduler cannot execute an unlimited number of tokens in a single iteration.

It must decide how much of the available budget is assigned to each active request.

---

### 3. num_computed_tokens

Each request tracks how much work has already been completed.

The important relationship is:

```text
remaining work
    =
tokens that should exist
    -
tokens already computed
```

Conceptually:

```text
num_new_tokens
    ≈
num_tokens_with_spec
    -
num_computed_tokens
```

This allows the scheduler to represent both prefill and decode using the same scheduling mechanism.

---

### 4. Continuous Batching

vLLM does not require a fixed batch to finish before admitting new requests.

For example:

```text
Step 1:
A B C

Step 2:
A C D

Step 3:
A D E
```

Requests can leave the batch when they finish, while new requests can enter when capacity becomes available.

This dynamic behavior is the basis of continuous batching.

---

### 5. Chunked Prefill

A long prompt does not necessarily need to complete its entire prefill in one scheduler step.

For example:

```text
8192-token prompt
    ↓
2048 tokens
    ↓
2048 tokens
    ↓
2048 tokens
    ↓
2048 tokens
```

This allows long-prefill requests to share scheduler steps with decode requests and other workloads.

The tradeoff is between:

```text
larger prefill chunks
→ higher prefill efficiency

smaller prefill chunks
→ better decode responsiveness
```

---

## Scheduler Mental Model

The high-level scheduler flow is:

```text
Scheduler.schedule()
        ↓
Create token budget
        ↓
Schedule RUNNING requests
        ↓
Calculate num_new_tokens
        ↓
Allocate KV-cache slots
        ↓
Schedule WAITING requests
        ↓
WAITING → RUNNING
        ↓
Record scheduled token counts
        ↓
Build SchedulerOutput
        ↓
Model execution
```

---

## Questions to Answer

By the end of Day 3, we should be able to answer:

1. What is the difference between `waiting` and `running`?
2. What limits the number of tokens scheduled in one step?
3. How is `num_new_tokens` calculated?
4. Why does vLLM not need completely separate prefill and decode schedulers?
5. How does continuous batching emerge from repeated calls to `Scheduler.schedule()`?
6. How does chunked prefill share compute with decode requests?
7. What happens when the token budget is exhausted?
8. What happens when KV-cache allocation fails?

---

## Expected Outcome

At the end of Day 3, we should be able to explain a scheduler iteration using a simple example such as:

```text
Request A:
long prefill, 3000 tokens remaining

Request B:
decode, 1 token needed

Request C:
waiting request

Token budget:
2048 tokens
```

A scheduler step may allocate:

```text
A → 2047 tokens
B → 1 token
C → remains waiting
```

The next scheduler step repeats the process using updated request state.

---

## Day 3 Summary Goal

**Understand how vLLM allocates per-step token budget across running and waiting requests, enabling continuous batching and chunked prefill.**


In [31]:
# =============================================================================
# Step 1 — Inspect Scheduler initialization
# =============================================================================

print_section("Step 1 — Inspect Scheduler initialization")

import inspect
from vllm.v1.core.sched.scheduler import Scheduler

# TODO:
# Inspect Scheduler.__init__().
#
# Questions:
#   1. How are waiting requests stored?
#   2. How are running requests stored?
#   3. Which field defines the per-step scheduling token budget?
#   4. Which field limits the number of concurrently running requests?
#

scheduler_init_method = Scheduler.__init__

# TODO:
# Find the source file.
scheduler_init_file = inspect.getfile(scheduler_init_method)

print("Scheduler.__init__() source file:")
print(scheduler_init_file)

# TODO:
# Print the source code.
scheduler_init_source = inspect.getsource(scheduler_init_method)
print(scheduler_init_source)


Step 1 — Inspect Scheduler initialization
Scheduler.__init__() source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/sched/scheduler.py
    def __init__(
        self,
        vllm_config: VllmConfig,
        kv_cache_config: KVCacheConfig,
        structured_output_manager: StructuredOutputManager,
        block_size: int,
        hash_block_size: int | None = None,
        mm_registry: MultiModalRegistry = MULTIMODAL_REGISTRY,
        include_finished_set: bool = False,
        log_stats: bool = False,
    ) -> None:
        self.vllm_config = vllm_config
        self.scheduler_config = vllm_config.scheduler_config
        self.cache_config = vllm_config.cache_config
        self.lora_config = vllm_config.lora_config
        self.model_uses_mrope = vllm_config.model_config.uses_mrope
        self.model_uses_xdrope = vllm_config.model_config.uses_xdrope
        self.kv_cache_config = kv_cache_config
        self.kv_events_config = vllm_config.kv_events_config
        self

In [32]:
# =============================================================================
# Step 2 — Inspect Request token state
# =============================================================================

print_section("Step 2 — Inspect Request token state")

import inspect
from vllm.v1.request import Request

# TODO:
# Inspect the Request class.
#
# Focus on these fields / properties:
#   - num_prompt_tokens
#   - num_computed_tokens
#   - num_tokens
#   - num_tokens_with_spec
#   - output_token_ids
#   - spec_token_ids
#
# Questions:
#   1. Which fields are stored directly?
#   2. Which values are computed properties?
#   3. What does each token count represent?
#

request_file = inspect.getfile(Request)

print("Request source file:")
print(request_file)

request_source = inspect.getsource(Request)

print(request_source)


Step 2 — Inspect Request token state
Request source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/request.py
class Request:
    def __init__(
        self,
        request_id: str,
        prompt_token_ids: list[int] | None,
        sampling_params: SamplingParams | None,
        pooling_params: PoolingParams | None,
        client_index: int = 0,
        arrival_time: float | None = None,
        prompt_embeds: torch.Tensor | None = None,
        prompt_is_token_ids: list[bool] | None = None,
        mm_features: list[MultiModalFeatureSpec] | None = None,
        lora_request: "LoRARequest | None" = None,
        cache_salt: str | None = None,
        priority: int = 0,
        trace_headers: Mapping[str, str] | None = None,
        block_hasher: Callable[["Request"], list["BlockHash"]] | None = None,
        resumable: bool = False,
        session_id: str | None = None,
        reasoning_ended: bool | None = None,
        reasoning_parser_kwargs: dict[str, Any] | None = Non

In [33]:
# =============================================================================
# Step 3 — Inspect token-budget clipping for RUNNING requests
# =============================================================================

print_section("Step 3 — Inspect token-budget clipping")

import inspect
from vllm.v1.core.sched.scheduler import Scheduler

# TODO:
# Inspect Scheduler.schedule() and focus only on the logic that computes
# num_new_tokens for RUNNING requests.
#
# Questions:
#   1. How is the initial num_new_tokens computed?
#   2. Which constraints can reduce it?
#   3. Where is the global token budget applied?
#

schedule_source = inspect.getsource(Scheduler.schedule)

# Print only lines related to token scheduling.
keywords = [
    "num_new_tokens",
    "token_budget",
    "input_budget",
    "long_prefill_token_threshold",
    "max_model_len",
]

for line in schedule_source.splitlines():
    if any(keyword in line for keyword in keywords):
        print(line)


Step 3 — Inspect token-budget clipping
        token_budget = self.max_num_scheduled_tokens
        input_budget = self.scheduler_config.max_num_batched_tokens
            token_budget = 0
        while req_index < len(self.running) and token_budget > 0:
            if input_budget <= draft_slots:
            num_new_tokens = (
            if 0 < self.scheduler_config.long_prefill_token_threshold < num_new_tokens:
                num_new_tokens = self.scheduler_config.long_prefill_token_threshold
            num_new_tokens = min(
                num_new_tokens, token_budget, input_budget - draft_slots
            num_new_tokens = min(
                num_new_tokens,
                self.max_model_len
                num_new_tokens = self._mamba_block_aligned_split(
                    request, num_new_tokens
                    num_new_tokens,
                    num_new_tokens,
            num_new_tokens = self._reserve_prefill_lookahead(
                request, request.num_computed

In [34]:
# =============================================================================
# Step 4 — Inspect WAITING -> RUNNING transition
# =============================================================================

print_section("Step 4 — Inspect WAITING -> RUNNING transition")

import inspect
from vllm.v1.core.sched.scheduler import Scheduler

schedule_source = inspect.getsource(Scheduler.schedule)

# TODO:
# Print only the section responsible for scheduling WAITING requests.
#
# Focus on these keywords:
#   - self.waiting
#   - request_token_budget
#   - num_computed_tokens
#   - num_new_tokens
#   - allocate_slots
#   - self.running.append
#   - RequestStatus.RUNNING
#
# Questions:
#   1. Under what conditions can a WAITING request be considered?
#   2. How is its token budget calculated?
#   3. When does KV-cache allocation happen?
#   4. What exact lines move the request into RUNNING?
#

keywords = [
    "self.waiting",
    "request_token_budget",
    "num_computed_tokens",
    "num_new_tokens",
    "allocate_slots",
    "self.running.append",
    "RequestStatus.RUNNING",
]

for line in schedule_source.splitlines():
    if any(keyword in line for keyword in keywords):
        print(line)


Step 4 — Inspect WAITING -> RUNNING transition
        # Each request just has the num_computed_tokens and
        # so that each request's num_computed_tokens can catch up its
                # This is (num_computed_tokens + 1) - (num_output_placeholders - 1).
                and request.num_computed_tokens + 2 - request.num_output_placeholders
            num_new_tokens = (
                - request.num_computed_tokens
            if 0 < self.scheduler_config.long_prefill_token_threshold < num_new_tokens:
                num_new_tokens = self.scheduler_config.long_prefill_token_threshold
            num_new_tokens = min(
                num_new_tokens, token_budget, input_budget - draft_slots
            num_new_tokens = min(
                num_new_tokens,
                - request.num_computed_tokens
                num_new_tokens = self._mamba_block_aligned_split(
                    request, num_new_tokens
                    num_new_tokens,
                    request.num_compu

In [35]:
# =============================================================================
# Step 5 — Inspect KV allocation failure and preemption
# =============================================================================

print_section("Step 5 — Inspect KV allocation failure and preemption")

import inspect
from vllm.v1.core.sched.scheduler import Scheduler

schedule_source = inspect.getsource(Scheduler.schedule)

# TODO:
# Inspect the RUNNING-request path around KV-cache allocation.
#
# Focus on:
#   - allocate_slots
#   - new_blocks
#   - preempt
#   - free
#   - token_budget
#   - input_budget
#
# Questions:
#   1. How does the scheduler detect KV-cache allocation failure?
#   2. What happens after allocation fails?
#   3. Which request gets preempted?
#   4. Why are token/input budgets restored in some branches?
#

keywords = [
    "allocate_slots",
    "new_blocks",
    "preempt",
    "_preempt",
    "free",
    "restored",
    "token_budget +=",
    "input_budget +=",
]

for line in schedule_source.splitlines():
    if any(keyword in line for keyword in keywords):
        print(line)


Step 5 — Inspect KV allocation failure and preemption
        preempted_reqs: list[Request] = []
        req_to_new_blocks: dict[str, KVCacheBlocks] = {}
            with record_function_or_nullcontext("schedule: allocate_slots"):
                    new_blocks = self.kv_cache_manager.allocate_slots(
                    if new_blocks is not None:
                        preempted_req = max(
                        # Record the index of the preemption victim to
                        victim_index = self.running.index(preempted_req)
                        if preempted_req in scheduled_running_reqs:
                            preempted_req_id = preempted_req.request_id
                            scheduled_running_reqs.remove(preempted_req)
                            restored = num_scheduled_tokens.pop(preempted_req_id)
                            token_budget += restored
                            input_budget += restored + draft_slots
                            req_to_new_blocks.

In [36]:
# =============================================================================
# Step 6 — Inspect post-execution request state updates
# =============================================================================

print_section("Step 6 — Inspect post-execution request state updates")

import inspect
from vllm.v1.core.sched.scheduler import Scheduler

# TODO:
# Inspect the method that processes model outputs after a scheduler step.
#
# Focus on:
#   - update_from_output / update_from_outputs
#   - num_computed_tokens
#   - num_scheduled_tokens
#   - output_token_ids
#   - append_output_token_ids
#
# Questions:
#   1. Where does num_computed_tokens increase?
#   2. Is it increased by the number of scheduled tokens?
#   3. When are newly generated output tokens appended?
#   4. How does the request become ready for the next scheduler step?
#

# First, inspect candidate methods on Scheduler.
for name in dir(Scheduler):
    if "update_from" in name:
        print(name)


Step 6 — Inspect post-execution request state updates
_update_from_kv_xfer_finished
update_from_output


In [37]:
# =============================================================================
# Step 6.1 — Inspect Scheduler.update_from_output
# =============================================================================

import inspect
from vllm.v1.core.sched.scheduler import Scheduler

update_source = inspect.getsource(Scheduler.update_from_output)

print(update_source)

    def update_from_output(
        self,
        scheduler_output: SchedulerOutput,
        model_runner_output: ModelRunnerOutput,
    ) -> dict[int, EngineCoreOutputs]:
        sampled_token_ids = model_runner_output.sampled_token_ids
        logprobs = model_runner_output.logprobs
        prompt_logprobs_dict = model_runner_output.prompt_logprobs_dict
        num_scheduled_tokens = scheduler_output.num_scheduled_tokens
        pooler_outputs = model_runner_output.pooler_output
        num_nans_in_logits = model_runner_output.num_nans_in_logits
        kv_connector_output = model_runner_output.kv_connector_output
        ec_connector_output = model_runner_output.ec_connector_output
        cudagraph_stats = model_runner_output.cudagraph_stats

        # Every GPU write enqueued by this and earlier steps has completed, so it is
        # safe to return deferred-free blocks to the pool.
        if self.defer_block_free and scheduler_output.total_num_scheduled_tokens > 0:
            s

In [38]:
# =============================================================================
# Step 6.2 — Find where num_computed_tokens is updated
# =============================================================================

print_section("Step 6.2 — Find num_computed_tokens updates")

import inspect
from vllm.v1.core.sched.scheduler import Scheduler

scheduler_source = inspect.getsource(Scheduler)

for i, line in enumerate(scheduler_source.splitlines()):
    if "num_computed_tokens" in line:
        print(f"{i:4d}: {line}")


Step 6.2 — Find num_computed_tokens updates
 328:             request.num_computed_tokens
 409:         num_computed_tokens: int,
 422:         remaining = request.num_tokens - num_computed_tokens - num_new_tokens
 431:         # Each request just has the num_computed_tokens and
 435:         # so that each request's num_computed_tokens can catch up its
 485:                 # This is (num_computed_tokens + 1) - (num_output_placeholders - 1).
 490:                 and request.num_computed_tokens + 2 - request.num_output_placeholders
 514:                 - request.num_computed_tokens
 527:                 - request.num_computed_tokens
 549:                     request.num_computed_tokens,
 558:                 request, request.num_computed_tokens, num_new_tokens
 661:                     + request.num_computed_tokens
 764:                 if request.num_computed_tokens == 0:
 834:                     num_computed_tokens = (
 837:                     assert num_computed_tokens <= reque

In [39]:
# =============================================================================
# Step 6.3 — Inspect the exact num_computed_tokens update logic
# =============================================================================

print_section("Step 6.3 — Inspect num_computed_tokens update logic")

import inspect
from vllm.v1.core.sched.scheduler import Scheduler

scheduler_source = inspect.getsource(Scheduler)
lines = scheduler_source.splitlines()

for i in range(1365, 1410):
    print(f"{i:4d}: {lines[i]}")


Step 6.3 — Inspect num_computed_tokens update logic
1365:         request.num_output_placeholders = 0
1366:         request.num_preemptions += 1
1367:         if self.log_stats:
1368:             request.record_event(EngineCoreEventType.PREEMPTED, timestamp)
1369: 
1370:         # Put the request back to the waiting queue.
1371:         self.waiting.prepend_request(request)
1372:         self.reset_preempted_req_ids.add(request.request_id)
1373: 
1374:     def _update_after_schedule(self, scheduler_output: SchedulerOutput) -> None:
1375:         # Advance the number of computed tokens for the request AFTER
1376:         # the request is scheduled.
1377:         # 1. The scheduler_output of the current step has to include the
1378:         #    original number of scheduled tokens to determine input IDs.
1379:         # 2. Advance the number of computed tokens here allowing us to
1380:         #    schedule the prefill request again immediately in the next
1381:         #    scheduling 

In [40]:
# =============================================================================
# Step 7 — Inspect chunked prefill control logic
# =============================================================================

print_section("Step 7 — Inspect chunked prefill control logic")

import inspect
from vllm.v1.core.sched.scheduler import Scheduler

schedule_source = inspect.getsource(Scheduler.schedule)

keywords = [
    "long_prefill_token_threshold",
    "chunked_prefill_enabled",
    "num_new_tokens",
    "request_token_budget",
    "is_prefill_chunk",
]

for i, line in enumerate(schedule_source.splitlines()):
    if any(keyword in line for keyword in keywords):
        print(f"{i:4d}: {line}")


Step 7 — Inspect chunked prefill control logic
  47:         ) and any(not r.is_prefill_chunk for r in self.running)
  78:             if defer_prefills and request.is_prefill_chunk:
  84:             num_new_tokens = (
  89:             if 0 < self.scheduler_config.long_prefill_token_threshold < num_new_tokens:
  90:                 num_new_tokens = self.scheduler_config.long_prefill_token_threshold
  91:             num_new_tokens = min(
  92:                 num_new_tokens, token_budget, input_budget - draft_slots
  97:             num_new_tokens = min(
  98:                 num_new_tokens,
 106:                 num_new_tokens = self._mamba_block_aligned_split(
 107:                     request, num_new_tokens
 117:                     num_new_tokens,
 123:                     num_new_tokens,
 130:             num_new_tokens = self._reserve_prefill_lookahead(
 131:                 request, request.num_computed_tokens, num_new_tokens
 134:             if num_new_tokens == 0:
 159:  

In [41]:
# =============================================================================
# Step 7.1 — Inspect chunked-prefill admission condition
# =============================================================================

print_section("Step 7.1 — Inspect chunked-prefill admission condition")

import inspect
from vllm.v1.core.sched.scheduler import Scheduler

schedule_source = inspect.getsource(Scheduler.schedule)
lines = schedule_source.splitlines()

for i in range(475, 502):
    print(f"{i:4d}: {lines[i]}")


Step 7.1 — Inspect chunked-prefill admission condition
 475:                             <= self.max_model_len
 476:                         ):
 477:                             if padded_num_tokens > request_token_budget:
 478:                                 # Prefer to not schedule than schedule un-padded.
 479:                                 break
 480:                             num_new_tokens = padded_num_tokens
 481:                             pad_spec_decode = True
 482: 
 483:                     threshold = self.scheduler_config.long_prefill_token_threshold
 484:                     if 0 < threshold < num_new_tokens:
 485:                         num_new_tokens = threshold
 486: 
 487:                     # chunked prefill has to be enabled explicitly to allow
 488:                     # pooling requests to be chunked
 489:                     if (
 490:                         not self.scheduler_config.enable_chunked_prefill
 491:                         and num_new_toke

## scheduler internals

```text
Request enters WAITING
    ↓
Scheduler.schedule()
    ↓
RUNNING requests first
    ↓
calculate remaining work
    ↓
apply long-prefill threshold
    ↓
apply global token/input budget
    ↓
allocate KV cache
    ↓
if KV insufficient:
    preempt / rollback
    ↓
use remaining capacity for WAITING
    ↓
WAITING → RUNNING
    ↓
_update_after_schedule()
    ↓
num_computed_tokens += scheduled tokens
    ↓
ModelRunner executes
    ↓
update_from_output()
    ↓
append generated tokens / rollback rejected spec tokens
    ↓
next scheduler iteration

# Day 4 — KV Cache, PagedAttention, and Block Allocation

## Objective

The goal of Day 4 is to understand how vLLM manages KV-cache memory at the source-code level.

Day 3 established that the scheduler does not only manage compute resources. Every scheduled request must also have enough KV-cache capacity.

The key path we observed was:

```text
Scheduler.schedule()
    ↓
calculate num_new_tokens
    ↓
kv_cache_manager.allocate_slots(...)
    ↓
KV allocation succeeds?
    ├── yes → request can continue
    └── no  → preemption may be required
```

Day 4 starts from `KVCacheManager.allocate_slots()` and follows the memory-management path downward.

The main question is:

> How does vLLM translate scheduled tokens into physical KV-cache blocks, and how does block-based allocation make dynamic LLM serving practical?

---

## Main Concepts

Day 4 focuses on six core concepts:

### 1. Why KV Cache Needs Memory Management

During autoregressive inference, previously computed keys and values are reused during future decode steps.

Conceptually:

```text
Prompt tokens
    ↓
compute K / V
    ↓
store in KV cache
    ↓
future decode tokens reuse them
```

As sequence length increases, the amount of KV-cache memory also increases.

For multiple concurrent requests:

```text
Request A → KV cache
Request B → KV cache
Request C → KV cache
Request D → KV cache
```

The serving system therefore needs to continuously allocate, reuse, and free GPU memory.

---

## 2. The Problem with Contiguous KV Allocation

A simple implementation could allocate one contiguous KV-cache region for every request.

Conceptually:

```text
Request A:
[ A A A A A A ]

Request B:
[ B B B ]

Request C:
[ C C C C ]
```

However, request lengths are dynamic.

For example:

```text
A grows
B finishes
C grows
D arrives
```

This can create fragmentation and makes it difficult to reserve large contiguous regions efficiently.

Conceptually:

```text
GPU KV memory

[A][A][free][B][B][free][C][C][C][free]
```

Even when total free memory is sufficient, finding a sufficiently large contiguous region may become difficult.

---

## 3. Paged KV Cache

vLLM manages KV cache using fixed-size blocks.

Conceptually:

```text
Logical sequence:

Token 0
Token 1
Token 2
...
Token N
```

is divided into logical blocks:

```text
Logical KV blocks

Block 0
[0 1 2 ... 15]

Block 1
[16 17 18 ... 31]

Block 2
[32 33 34 ... 47]
```

These logical blocks do not have to be physically adjacent in GPU memory.

For example:

```text
Request A logical blocks

Logical Block 0 → Physical Block 7
Logical Block 1 → Physical Block 21
Logical Block 2 → Physical Block 4
```

Conceptually:

```text
Request A

Logical:
[0] → [1] → [2]

Physical GPU KV cache:
       ↓     ↓     ↓
      [7]   [21]   [4]
```

This is the basic memory-management idea behind PagedAttention.

---

## 4. Token Scheduling vs KV Allocation

Day 3 showed that the scheduler first decides how many tokens a request should compute:

```text
num_new_tokens
```

But the scheduler still needs enough KV-cache space for those tokens.

Therefore:

```text
Scheduler
    ↓
num_new_tokens = N
    ↓
KVCacheManager.allocate_slots(
    request,
    N,
    ...
)
```

The scheduler is effectively asking:

> Can the KV-cache manager provide enough physical storage for the next N token positions?

If allocation succeeds:

```text
token budget available
+
KV capacity available
=
request can execute
```

If allocation fails:

```text
token budget available
+
KV capacity unavailable
=
request may still be unable to execute
```

This is why LLM scheduling combines compute scheduling and memory scheduling.

---

## 5. KV Blocks and Request Growth

A request does not necessarily allocate all of its future KV-cache memory at admission time.

Instead, KV capacity grows as the request progresses.

For example:

```text
Block size = 16 tokens
```

A request with a 30-token prompt may need:

```text
Block 0 → tokens 0–15
Block 1 → tokens 16–29
```

After additional decode tokens are generated:

```text
Token 30
Token 31
Token 32
...
```

new KV slots may be required.

Conceptually:

```text
Step 1:
[A0][A1]

Step 2:
[A0][A1][A2]

Step 3:
[A0][A1][A2][A3]
```

The KV-cache manager therefore participates in almost every scheduler iteration.

---

## 6. KV Allocation Failure and Preemption

Day 3 already showed the first important failure path:

```text
allocate_slots(...)
    ↓
returns None
    ↓
insufficient KV capacity
```

For a running request, the scheduler may then preempt another request:

```text
KV allocation failure
    ↓
choose victim
    ↓
preempt victim
    ↓
release / reset its KV state
    ↓
retry allocation
```

This produces a tradeoff:

```text
Higher concurrency
    ↓
higher KV pressure
    ↓
greater chance of preemption
    ↓
possible recomputation
    ↓
higher latency / lower throughput
```

Understanding this relationship will be important for the later benchmark.

---

# Day 4 Source-Code Path

The main source-code path to investigate is:

```text
Scheduler.schedule()
    ↓
KVCacheManager.allocate_slots()
    ↓
determine required KV positions
    ↓
determine required blocks
    ↓
allocate blocks from block pool
    ↓
associate blocks with request
```

We will investigate these components step by step:

```text
Scheduler
    ↓
KVCacheManager
    ↓
KVCacheCoordinator / cache groups
    ↓
BlockPool
    ↓
KVCacheBlock
```

The exact classes may vary slightly depending on the current vLLM implementation, so we will follow the source rather than assuming the internal architecture.

---

# Questions to Answer

By the end of Day 4, we should be able to answer:

1. What does `KVCacheManager.allocate_slots()` actually do?
2. What is a KV-cache block in vLLM?
3. How many tokens can one block hold?
4. How does a request keep track of its allocated blocks?
5. How are new blocks allocated as a request grows?
6. What happens to blocks when a request finishes?
7. Why can physical KV blocks be non-contiguous?
8. How does this enable PagedAttention?
9. How does prefix caching reuse existing KV blocks?
10. What exactly causes `allocate_slots()` to return `None`?
11. How is KV pressure related to scheduler preemption?
12. What metrics could we collect to measure KV-cache pressure?

---

# Mental Model

A useful mental model for Day 4 is:

```text
Request
    ↓
logical token positions
    ↓
logical KV blocks
    ↓
block table / request block mapping
    ↓
physical KV blocks
    ↓
GPU KV-cache memory
```

The scheduler operates primarily in token space:

```text
"Request A should compute 512 more tokens."
```

The KV-cache manager converts this decision into memory resources:

```text
"Those 512 positions require these additional KV blocks."
```

The model runner then uses the resulting block mapping to access KV memory during attention.

---

# Example

Assume:

```text
block_size = 16 tokens
```

Request A currently ha


In [42]:
# =============================================================================
# Step 1 — Inspect KVCacheManager.allocate_slots
# =============================================================================

print_section("Step 1 — Inspect KVCacheManager.allocate_slots")

import inspect
from vllm.v1.core.kv_cache_manager import KVCacheManager

# TODO:
# Inspect KVCacheManager.allocate_slots().
#
# Questions:
#   1. What inputs does allocate_slots() receive?
#   2. How does it determine how many new KV slots / blocks are required?
#   3. Under what condition does it return None?
#   4. What does it return on success?
#

allocate_slots_method = KVCacheManager.allocate_slots

kv_cache_manager_file = inspect.getfile(KVCacheManager)
allocate_slots_source = inspect.getsource(allocate_slots_method)

print("KVCacheManager source file:")
print(kv_cache_manager_file)

print("\nKVCacheManager.allocate_slots() source:")
print(allocate_slots_source)


Step 1 — Inspect KVCacheManager.allocate_slots
KVCacheManager source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/kv_cache_manager.py

KVCacheManager.allocate_slots() source:
    def allocate_slots(
        self,
        request: Request,
        num_new_tokens: int,
        num_new_computed_tokens: int = 0,
        new_computed_blocks: KVCacheBlocks | None = None,
        num_lookahead_tokens: int = 0,
        num_external_computed_tokens: int = 0,
        delay_cache_blocks: bool = False,
        num_encoder_tokens: int = 0,
        full_sequence_must_fit: bool = False,
        reserved_blocks: int = 0,
        has_scheduled_reqs: bool = True,
    ) -> KVCacheBlocks | None:
        """Add slots for a request with new tokens to append.

        Args:
            request: The request to allocate slots.
            num_new_tokens: The number of new tokens to be allocated and computed.
            num_new_computed_tokens: The number of new computed tokens just
            

In [43]:
# =============================================================================
# Step 2 — Inspect get_num_blocks_to_allocate
# =============================================================================

print_section("Step 2 — Inspect get_num_blocks_to_allocate")

import inspect

# We already have a KVCacheManager instance/class path from Step 1.
# Now inspect the coordinator method used by allocate_slots().

from vllm.v1.core.kv_cache_manager import KVCacheManager

# TODO:
# Find the coordinator class from the KVCacheManager implementation.
# Then inspect get_num_blocks_to_allocate().
#
# Questions:
#   1. Which class implements get_num_blocks_to_allocate()?
#   2. How does it convert num_tokens into required blocks?
#   3. Does it account for already allocated blocks?
#   4. How does block_size enter the calculation?
#   5. Does it aggregate requirements across multiple KV-cache groups?
#

manager_source = inspect.getsource(KVCacheManager)

for i, line in enumerate(manager_source.splitlines()):
    if "coordinator" in line or "get_num_blocks_to_allocate" in line:
        print(f"{i:4d}: {line}")


Step 2 — Inspect get_num_blocks_to_allocate
  35:         self.coordinator = get_kv_cache_coordinator(
  50:         self.block_pool = self.coordinator.block_pool
 142:             self.coordinator.find_longest_cache_hit(
 199:         coordinator = self.coordinator
 202:             and isinstance(coordinator, HybridKVCacheCoordinator)
 203:             and coordinator.full_attention_group_id is not None
 210:         fa_group_id = coordinator.full_attention_group_id
 211:         computed, per_group_hits = coordinator.find_longest_cache_hit_per_group(
 357:             num_blocks_to_allocate = self.coordinator.get_num_blocks_to_allocate(
 385:         self.coordinator.remove_skipped_blocks(
 391:         num_blocks_to_allocate = self.coordinator.get_num_blocks_to_allocate(
 416:             self.coordinator.allocate_new_computed_blocks(
 423:         new_blocks = self.coordinator.allocate_new_blocks(
 444:         self.coordinator.cache_blocks(request, num_tokens_to_cache)
 456:    

In [44]:
# =============================================================================
# Step 2.1 — Inspect coordinator factory
# =============================================================================

print_section("Step 2.1 — Inspect coordinator factory")

import inspect
from vllm.v1.core.kv_cache_manager import get_kv_cache_coordinator

print("Factory source file:")
print(inspect.getfile(get_kv_cache_coordinator))

print("\nFactory source:")
print(inspect.getsource(get_kv_cache_coordinator))


Step 2.1 — Inspect coordinator factory
Factory source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/kv_cache_coordinator.py

Factory source:
def get_kv_cache_coordinator(
    kv_cache_config: KVCacheConfig,
    max_model_len: int,
    max_in_flight_tokens: int,
    use_eagle: bool,
    enable_caching: bool,
    enable_kv_cache_events: bool,
    dcp_world_size: int,
    pcp_world_size: int,
    scheduler_block_size: int,
    hash_block_size: int,
    metrics_collector: KVCacheMetricsCollector | None = None,
    num_prefill_lookahead: int = 0,
) -> KVCacheCoordinator:
    if not enable_caching:
        return KVCacheCoordinatorNoPrefixCache(
            kv_cache_config,
            max_model_len,
            max_in_flight_tokens,
            use_eagle,
            enable_kv_cache_events,
            dcp_world_size=dcp_world_size,
            pcp_world_size=pcp_world_size,
            scheduler_block_size=scheduler_block_size,
            hash_block_size=hash_block_size,
   

In [45]:
# =============================================================================
# Step 2.2 — Find get_num_blocks_to_allocate implementations
# =============================================================================

print_section("Step 2.2 — Find get_num_blocks_to_allocate implementations")

import inspect
import vllm.v1.core.kv_cache_coordinator as kv_coord

classes_to_check = [
    kv_coord.KVCacheCoordinatorNoPrefixCache,
    kv_coord.UnitaryKVCacheCoordinator,
    kv_coord.HybridKVCacheCoordinator,
]

for cls in classes_to_check:
    print("\n" + "=" * 80)
    print(cls.__name__)
    print("=" * 80)

    if hasattr(cls, "get_num_blocks_to_allocate"):
        method = cls.get_num_blocks_to_allocate

        print("Defined in:")
        print(inspect.getfile(method))

        print("\nSource:")
        print(inspect.getsource(method))


Step 2.2 — Find get_num_blocks_to_allocate implementations

KVCacheCoordinatorNoPrefixCache
Defined in:
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/kv_cache_coordinator.py

Source:
    def get_num_blocks_to_allocate(
        self,
        request_id: str,
        num_tokens: int,
        new_computed_blocks: tuple[Sequence[KVCacheBlock], ...],
        num_encoder_tokens: int,
        total_computed_tokens: int,
        num_local_computed_tokens: int,
        num_tokens_main_model: int,
        apply_admission_cap: bool = False,
    ) -> int:
        """
        Get the number of blocks needed to be allocated for the request.

        Args:
            request_id: The request ID.
            num_tokens: The total number of tokens that need a slot (including
                tokens that are already allocated).
            new_computed_blocks: The new computed blocks just hitting the
                prefix caching.
            num_encoder_tokens: The number of encoder tokens for 

In [46]:
# =============================================================================
# Step 2.3 — Find single-type KV cache managers
# =============================================================================

print_section("Step 2.3 — Inspect single-type KV cache managers")

import inspect
import vllm.v1.core.kv_cache_coordinator as kv_coord

# Find classes in this module that implement get_num_blocks_to_allocate().
for name, obj in inspect.getmembers(kv_coord, inspect.isclass):

    if hasattr(obj, "get_num_blocks_to_allocate"):

        method = obj.get_num_blocks_to_allocate

        # Only print classes that actually define the method themselves,
        # instead of merely inheriting it.
        if "get_num_blocks_to_allocate" in obj.__dict__:

            print("\n" + "=" * 80)
            print(name)
            print("=" * 80)

            print(inspect.getsource(method))


Step 2.3 — Inspect single-type KV cache managers

KVCacheCoordinator
    def get_num_blocks_to_allocate(
        self,
        request_id: str,
        num_tokens: int,
        new_computed_blocks: tuple[Sequence[KVCacheBlock], ...],
        num_encoder_tokens: int,
        total_computed_tokens: int,
        num_local_computed_tokens: int,
        num_tokens_main_model: int,
        apply_admission_cap: bool = False,
    ) -> int:
        """
        Get the number of blocks needed to be allocated for the request.

        Args:
            request_id: The request ID.
            num_tokens: The total number of tokens that need a slot (including
                tokens that are already allocated).
            new_computed_blocks: The new computed blocks just hitting the
                prefix caching.
            num_encoder_tokens: The number of encoder tokens for allocating
                blocks for cross-attention.
            total_computed_tokens: Include both local and external

In [47]:
# =============================================================================
# Step 3 — Inspect physical block allocation
# =============================================================================

print_section("Step 3 — Inspect physical block allocation")

import inspect
import vllm.v1.core.kv_cache_coordinator as kv_coord

# TODO:
# Find implementations of allocate_new_blocks().
#
# Questions:
#   1. Which class actually allocates new physical KV blocks?
#   2. How many blocks are requested from the block pool?
#   3. How are allocated blocks associated with request_id?
#   4. Does the method directly manipulate the free-block pool?
#

for name, obj in inspect.getmembers(kv_coord, inspect.isclass):
    if (
        hasattr(obj, "allocate_new_blocks")
        and "allocate_new_blocks" in obj.__dict__
    ):
        print("\n" + "=" * 80)
        print(name)
        print("=" * 80)
        print(inspect.getsource(obj.allocate_new_blocks))


Step 3 — Inspect physical block allocation

KVCacheCoordinator
    def allocate_new_blocks(
        self,
        request_id: str,
        num_tokens: int,
        num_tokens_main_model: int,
        num_encoder_tokens: int = 0,
    ) -> tuple[list[KVCacheBlock], ...]:
        """
        Allocate new blocks for the request to give it at least `num_tokens`
        token slots.

        Args:
            request_id: The request ID.
            num_tokens: The total number of tokens that need a slot (including
                tokens that are already allocated).
            num_tokens_main_model: The number of tokens for the main model (aka target
                model in spec decode). w/o spec decode, it is num_tokens;
                with spec decode, it is num_tokens - num_lookahead_tokens.
            num_encoder_tokens: The number of encoder tokens for allocating
                blocks for cross-attention.

        Returns:
            The new allocated blocks.
        """
        r

In [48]:
# =============================================================================
# Step 3.1 — Inspect BlockPool.get_new_blocks
# =============================================================================

print_section("Step 3.1 — Inspect BlockPool.get_new_blocks")

import inspect
import vllm.v1.core.block_pool as block_pool_module

# TODO:
# Inspect BlockPool and get_new_blocks().
#
# Questions:
#   1. What data structure stores free blocks?
#   2. How are blocks removed from the free pool?
#   3. How is ref_cnt updated?
#   4. What happens if the caller asks for more blocks than are free?
#

BlockPool = block_pool_module.BlockPool

print("BlockPool source file:")
print(inspect.getfile(BlockPool))

print("\nBlockPool.get_new_blocks() source:")
print(inspect.getsource(BlockPool.get_new_blocks))


Step 3.1 — Inspect BlockPool.get_new_blocks
BlockPool source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/block_pool.py

BlockPool.get_new_blocks() source:
    def get_new_blocks(self, num_blocks: int) -> list[KVCacheBlock]:
        """Get new blocks from the free block pool.

        Note that we do not check block cache in this function.

        Args:
            num_blocks: The number of blocks to allocate.

        Returns:
            A list of new block.
        """
        if num_blocks > self.get_num_free_blocks():
            raise ValueError(f"Cannot get {num_blocks} free blocks from the pool")

        ret: list[KVCacheBlock] = self.free_block_queue.popleft_n(num_blocks)

        # In order to only iterate the list once, we duplicated code a bit
        if self.enable_caching:
            for block in ret:
                self._maybe_evict_cached_block(block)
                assert block.ref_cnt == 0
                block.ref_cnt += 1
                if self.

In [49]:
# =============================================================================
# Step 3.2 — Inspect KVCacheBlock and BlockPool initialization
# =============================================================================

print_section("Step 3.2 — Inspect KVCacheBlock and BlockPool initialization")

import inspect
import vllm.v1.core.block_pool as block_pool_module

BlockPool = block_pool_module.BlockPool
KVCacheBlock = block_pool_module.KVCacheBlock

print("\n" + "=" * 80)
print("KVCacheBlock")
print("=" * 80)

print(inspect.getsource(KVCacheBlock))

print("\n" + "=" * 80)
print("BlockPool.__init__")
print("=" * 80)

print(inspect.getsource(BlockPool.__init__))


Step 3.2 — Inspect KVCacheBlock and BlockPool initialization

KVCacheBlock
@dataclass(slots=True)
class KVCacheBlock:
    """KV-cache block metadata."""

    # Block ID, ranging from 0 to num_gpu_blocks - 1.
    block_id: int
    # Reference count.
    ref_cnt: int = 0
    # The hash key (block hash + group id) of the block, only available
    # when the block is full and cached.
    _block_hash: BlockHashWithGroupId | None = None
    # Number of prefix tokens covered by _block_hash. For full blocks this is
    # the full block boundary; partial entries can end inside a cache block.
    _block_hash_num_tokens: int | None = None

    # Used to construct a doubly linked list for free blocks.
    # These two attributes should only be manipulated by FreeKVCacheBlockQueue.
    prev_free_block: "KVCacheBlock | None" = None
    next_free_block: "KVCacheBlock | None" = None

    # Whether the block is a null block that should never be cached.
    is_null: bool = False

    @property
    def b

In [50]:
# =============================================================================
# Step 4 — Inspect KV block freeing
# =============================================================================

print_section("Step 4 — Inspect KV block freeing")

import inspect
import vllm.v1.core.kv_cache_coordinator as kv_coord
import vllm.v1.core.block_pool as block_pool_module

# ---------------------------------------------------------------------------
# Part 1 — Find SingleTypeKVCacheManager.free()
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("SingleTypeKVCacheManager.free")
print("=" * 80)

print(inspect.getsource(kv_coord.SingleTypeKVCacheManager.free))

# ---------------------------------------------------------------------------
# Part 2 — Inspect BlockPool freeing logic
# ---------------------------------------------------------------------------

BlockPool = block_pool_module.BlockPool

for name in dir(BlockPool):
    if "free" in name.lower():
        print(name)


Step 4 — Inspect KV block freeing

SingleTypeKVCacheManager.free
    def free(self, request_id: str) -> None:
        """
        Free the blocks for the request.

        Args:
            request_id: The request ID.
        """
        # Free blocks in reverse order so that the tail blocks are freed first.
        self.block_pool.free_blocks(reversed(self.pop_blocks_for_free(request_id)))

free_blocks
get_num_free_blocks


In [51]:
# =============================================================================
# Step 4.1 — Inspect BlockPool.free_blocks
# =============================================================================

print_section("Step 4.1 — Inspect BlockPool.free_blocks")

import inspect
import vllm.v1.core.block_pool as block_pool_module

BlockPool = block_pool_module.BlockPool

print(inspect.getsource(BlockPool.free_blocks))


Step 4.1 — Inspect BlockPool.free_blocks
    def free_blocks(self, ordered_blocks: Iterable[KVCacheBlock]) -> None:
        """Free a list of blocks. The blocks should be ordered by their
        eviction priority, where the first block will be evicted first.

        Args:
            ordered_blocks: A list of blocks to free ordered by their eviction
                priority.
        """
        # Identify blocks with hash (LRU cache) and without it (never match APC)
        blocks_to_evict_last = []
        blocks_to_evict_first = []
        for block in ordered_blocks:
            block.ref_cnt -= 1
            if block.ref_cnt == 0 and not block.is_null:
                if block.block_hash is None or not self.enable_caching:
                    # LIFO reuse of non-cached blocks for better GPU locality.
                    blocks_to_evict_first.append(block)
                else:
                    # FIFO reuse of cached blocks for LRU eviction behavior.
                    blocks

In [53]:
# =============================================================================
# Step 5 — Inspect prefix-cache lookup
# =============================================================================

print_section("Step 5 — Inspect prefix-cache lookup")

import inspect
import vllm.v1.core.kv_cache_coordinator as kv_coord

classes_to_check = [
    kv_coord.KVCacheCoordinator,
    kv_coord.KVCacheCoordinatorNoPrefixCache,
    kv_coord.UnitaryKVCacheCoordinator,
    kv_coord.HybridKVCacheCoordinator,
]

for cls in classes_to_check:
    print("\n" + "=" * 80)
    print(cls.__name__)
    print("=" * 80)

    if hasattr(cls, "find_longest_cache_hit"):
        method = cls.find_longest_cache_hit
        print(inspect.getsource(method))


Step 5 — Inspect prefix-cache lookup

KVCacheCoordinator
    @abstractmethod
    def find_longest_cache_hit(
        self,
        block_hashes: list[BlockHash],
        max_cache_hit_length: int,
    ) -> tuple[tuple[list[KVCacheBlock], ...], int, int]:
        """Returns the per-group hit blocks, the hit length, and the number of
        ``num_uncached_common_prefix_tokens`` (a shared prefix that a
        sparse-retention group has not cached yet; 0 unless hybrid)."""
        pass


KVCacheCoordinatorNoPrefixCache
    def find_longest_cache_hit(
        self,
        block_hashes: list[BlockHash],
        max_cache_hit_length: int,
    ) -> tuple[tuple[list[KVCacheBlock], ...], int, int]:
        blocks: tuple[list[KVCacheBlock], ...] = tuple(
            [] for _ in range(self.num_single_type_manager)
        )
        return blocks, 0, 0


UnitaryKVCacheCoordinator
    def find_longest_cache_hit(
        self,
        block_hashes: list[BlockHash],
        max_cache_hit_length: i

In [54]:
# =============================================================================
# Step 5.1 — Inspect single-type prefix-cache lookup
# =============================================================================

print_section("Step 5.1 — Inspect single-type prefix-cache lookup")

import inspect
import vllm.v1.core.kv_cache_utils as kv_utils
import vllm.v1.core.kv_cache_coordinator as kv_coord

# Find classes that define find_longest_cache_hit themselves.
for module in [kv_coord, kv_utils]:
    for name, obj in inspect.getmembers(module, inspect.isclass):
        if (
            hasattr(obj, "find_longest_cache_hit")
            and "find_longest_cache_hit" in obj.__dict__
        ):
            print("\n" + "=" * 80)
            print(name)
            print("=" * 80)
            print(inspect.getsource(obj.find_longest_cache_hit))


Step 5.1 — Inspect single-type prefix-cache lookup

CrossAttentionManager
    @classmethod
    def find_longest_cache_hit(
        cls,
        block_hashes: BlockHashList,
        max_length: int,
        kv_cache_group_ids: list[int],
        block_pool: BlockPool,
        kv_cache_spec: KVCacheSpec,
        drop_eagle_block: bool,
        alignment_tokens: int,
        dcp_world_size: int = 1,
        pcp_world_size: int = 1,
    ) -> tuple[tuple[list[KVCacheBlock], ...], int]:
        assert isinstance(kv_cache_spec, CrossAttentionSpec), (
            "CrossAttentionManager can only be used for cross-attention groups"
        )
        # Cross-attention does not benefit from prefix caching since:
        # 1. Encoder states are unique per request (different audio/image
        #    inputs)
        # 2. Encoder states are computed once per request, not incrementally
        # 3. No reusable prefix exists between different multimodal inputs
        # Return empty blocks to indicate 

In [55]:
# =============================================================================
# Step 5.2 — Find concrete SingleTypeKVCacheManager subclasses
# =============================================================================

print_section("Step 5.2 — Find concrete KV-cache manager subclasses")

import inspect
from vllm.v1.core.kv_cache_coordinator import SingleTypeKVCacheManager


def print_subclasses(cls, indent=0):
    for subcls in cls.__subclasses__():
        print(" " * indent + subcls.__name__)
        print(" " * indent + "Module:", subcls.__module__)

        if "find_longest_cache_hit" in subcls.__dict__:
            print(" " * indent + "Defines find_longest_cache_hit: YES")
        else:
            print(" " * indent + "Defines find_longest_cache_hit: NO")

        print()

        print_subclasses(subcls, indent + 4)


print_subclasses(SingleTypeKVCacheManager)


Step 5.2 — Find concrete KV-cache manager subclasses
FullAttentionManager
Module: vllm.v1.core.single_type_kv_cache_manager
Defines find_longest_cache_hit: YES

    RSWAManager
    Module: vllm.v1.core.single_type_kv_cache_manager
    Defines find_longest_cache_hit: NO

    CircularBufferManager
    Module: vllm.v1.core.single_type_kv_cache_manager
    Defines find_longest_cache_hit: YES

    SinkFullAttentionManager
    Module: vllm.v1.core.single_type_kv_cache_manager
    Defines find_longest_cache_hit: NO

SlidingWindowManager
Module: vllm.v1.core.single_type_kv_cache_manager
Defines find_longest_cache_hit: YES

ChunkedLocalAttentionManager
Module: vllm.v1.core.single_type_kv_cache_manager
Defines find_longest_cache_hit: YES

MambaManager
Module: vllm.v1.core.single_type_kv_cache_manager
Defines find_longest_cache_hit: YES

CrossAttentionManager
Module: vllm.v1.core.single_type_kv_cache_manager
Defines find_longest_cache_hit: YES



In [56]:
# =============================================================================
# Step 5.3 — Inspect FullAttentionManager.find_longest_cache_hit
# =============================================================================

print_section("Step 5.3 — Inspect FullAttentionManager prefix-cache lookup")

import inspect
from vllm.v1.core.single_type_kv_cache_manager import FullAttentionManager

method = FullAttentionManager.find_longest_cache_hit

print("Source file:")
print(inspect.getfile(method))

print("\nSource:")
print(inspect.getsource(method))


Step 5.3 — Inspect FullAttentionManager prefix-cache lookup
Source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/core/single_type_kv_cache_manager.py

Source:
    @classmethod
    def find_longest_cache_hit(
        cls,
        block_hashes: BlockHashList,
        max_length: int,
        kv_cache_group_ids: list[int],
        block_pool: BlockPool,
        kv_cache_spec: KVCacheSpec,
        drop_eagle_block: bool,
        alignment_tokens: int,
        dcp_world_size: int = 1,
        pcp_world_size: int = 1,
    ) -> tuple[tuple[list[KVCacheBlock], ...], int]:
        assert isinstance(
            kv_cache_spec, FullAttentionSpec | ChunkedLocalAttentionSpec
        ), (
            "FullAttentionManager can only be used for full attention "
            "and chunked local attention groups"
        )
        block_size = kv_cache_spec.block_size
        if dcp_world_size > 1:
            # DCP shards each block's KV across ranks; hashes must be viewed at
            # the s

In [57]:
# =============================================================================
# Step 5.4 — Inspect BlockPool.get_cached_block
# =============================================================================

print_section("Step 5.4 — Inspect cached-block lookup")

import inspect
from vllm.v1.core.block_pool import BlockPool

print(inspect.getsource(BlockPool.get_cached_block))


Step 5.4 — Inspect cached-block lookup
    def get_cached_block(
        self, block_hash: BlockHash, kv_cache_group_ids: list[int]
    ) -> list[KVCacheBlock] | None:
        """Get the cached block by the block hash for each group in
        `kv_cache_group_ids`, or None if cache miss for any group.
        If there are duplicated blocks, we return the first block in the cache.

        Args:
            block_hash: The hash value of the block.
            kv_cache_group_ids: The ids of the KV cache groups.

        Returns:
            The cached blocks if exists, or None.
        """
        cached_blocks = []
        for group_id in kv_cache_group_ids:
            block_hash_with_group_id = make_block_hash_with_group_id(
                block_hash, group_id
            )
            block = self.cached_block_hash_to_block.get_one_block(
                block_hash_with_group_id
            )
            if not block:
                return None
            cached_blocks.append(blo

In [58]:
# =============================================================================
# Step 6 — Trace KV block IDs into SchedulerOutput / ModelRunner input
# =============================================================================

print_section("Step 6 — Trace KV block IDs into execution")

import inspect
from vllm.v1.core.sched.scheduler import Scheduler

schedule_source = inspect.getsource(Scheduler.schedule)

# TODO:
# Search Scheduler.schedule() for the structures that carry KV block IDs
# from KVCacheManager into SchedulerOutput.
#
# Questions:
#   1. Where are block IDs extracted from KVCacheBlocks?
#   2. Which request-level structure stores them?
#   3. Are block IDs included for both new and cached/running requests?
#   4. Which field in SchedulerOutput ultimately carries this information?
#

keywords = [
    "req_to_new_blocks",
    "get_block_ids",
    "block_ids",
    "NewRequestData",
    "CachedRequestData",
    "scheduled_new_reqs",
    "scheduled_cached_reqs",
    "SchedulerOutput",
]

for i, line in enumerate(schedule_source.splitlines()):
    if any(keyword in line for keyword in keywords):
        print(f"{i:4d}: {line}")


Step 6 — Trace KV block IDs into execution
   0:     def schedule(self, throttle_prefills: bool = False) -> SchedulerOutput:
  13:         scheduled_new_reqs: list[Request] = []
  18:         req_to_new_blocks: dict[str, KVCacheBlocks] = {}
 190:                             req_to_new_blocks.pop(preempted_req_id)
 224:             req_to_new_blocks[request_id] = new_blocks
 649:                         self._skip_zero_block_ids.update(
 650:                             self.kv_cache_manager.get_zeroing_block_ids_in_range(
 667:                     scheduled_new_reqs.append(request)
 675:                 req_to_new_blocks[request_id] = self.kv_cache_manager.get_blocks(
 726:         assert len(scheduled_new_reqs) + len(scheduled_resumed_reqs) + len(
 742:             scheduled_new_reqs.extend(scheduled_resumed_reqs)
 745:                 NewRequestData.from_request(
 747:                     req_to_new_blocks[req.request_id].get_block_ids(),
 752:                 for req in scheduled_n

In [59]:
# =============================================================================
# Step 6.1 — Inspect request data passed to ModelRunner
# =============================================================================

print_section("Step 6.1 — Inspect NewRequestData and CachedRequestData")

import inspect
import vllm.v1.core.sched.output as sched_output_module

classes_to_check = [
    sched_output_module.NewRequestData,
    sched_output_module.CachedRequestData,
    sched_output_module.SchedulerOutput,
]

for cls in classes_to_check:
    print("\n" + "=" * 80)
    print(cls.__name__)
    print("=" * 80)
    print(inspect.getsource(cls))


Step 6.1 — Inspect NewRequestData and CachedRequestData

NewRequestData
@dataclass
class NewRequestData:
    req_id: str
    prompt_token_ids: list[int] | None
    mm_features: list[MultiModalFeatureSpec]
    sampling_params: SamplingParams | None
    pooling_params: PoolingParams | None
    block_ids: tuple[list[int], ...]
    num_computed_tokens: int
    lora_request: LoRARequest | None
    prompt_embeds: "torch.Tensor | None" = None
    prompt_is_token_ids: list[bool] | None = None

    # Only used for v2 model runner.
    prefill_token_ids: list[int] | None = None

    @classmethod
    def from_request(
        cls,
        request: Request,
        block_ids: tuple[list[int], ...],
        prefill_token_ids: list[int] | None = None,
        uses_mrope: bool = False,
        uses_xdrope: bool = False,
    ) -> "NewRequestData":
        return cls(
            req_id=request.request_id,
            prompt_token_ids=request.prompt_token_ids,
            mm_features=strip_covered_mm_

In [60]:
# =============================================================================
# Step 6.2 — Find where ModelRunner consumes block IDs
# =============================================================================

print_section("Step 6.2 — Find ModelRunner block-table updates")

import inspect
import vllm.v1.worker.gpu_model_runner as gpu_runner_module

GpuModelRunner = gpu_runner_module.GPUModelRunner

runner_source = inspect.getsource(GpuModelRunner)

keywords = [
    "scheduled_new_reqs",
    "scheduled_cached_reqs",
    "new_block_ids",
    "block_ids",
    "block_table",
    "resumed_req_ids",
]

for i, line in enumerate(runner_source.splitlines()):
    if any(keyword in line for keyword in keywords):
        print(f"{i:4d}: {line}")


Step 6.2 — Find ModelRunner block-table updates
 650:         """One-time precomputation for _zero_block_ids.
 662:     def _zero_block_ids(self, block_ids: list[int]) -> None:
 665:             self._kv_block_zeroer.zero_block_ids(block_ids)
 729:         if scheduler_output.new_block_ids_to_zero:
 730:             self._zero_block_ids(scheduler_output.new_block_ids_to_zero)
 748:         resumed_req_ids = scheduler_output.scheduled_cached_reqs.resumed_req_ids
 749:         # NOTE(zhuohan): cached_req_ids and resumed_req_ids are usually disjoint,
 750:         # so `(scheduled_req_ids - resumed_req_ids) == scheduled_req_ids` holds
 752:         # that case we include the resumed_req_ids in the unscheduled set so
 755:         unscheduled_req_ids = cached_req_ids - (scheduled_req_ids - resumed_req_ids)
 774:         for new_req_data in scheduler_output.scheduled_new_reqs:
 812:                 block_ids=new_req_data.block_ids,
 842:         req_data = scheduler_output.scheduled_cached

In [61]:
# =============================================================================
# Step 6.3 — Inspect ModelRunner request/block-table state update
# =============================================================================

print_section("Step 6.3 — Inspect ModelRunner block-table state update")

import inspect
import vllm.v1.worker.gpu_model_runner as gpu_runner_module

GpuModelRunner = gpu_runner_module.GPUModelRunner

runner_source = inspect.getsource(GpuModelRunner)
lines = runner_source.splitlines()

for i in range(760, 995):
    print(f"{i:4d}: {lines[i]}")


Step 6.3 — Inspect ModelRunner block-table state update
 760:         for req_id in unscheduled_req_ids:
 761:             self.input_batch.remove_request(req_id)
 762: 
 763:         is_ngram_gpu = (
 764:             self.speculative_config is not None
 765:             and self.speculative_config.use_ngram_gpu()
 766:         )
 767:         if is_ngram_gpu:
 768:             ngram_gpu_new_reqs: list[CachedRequestState] = []
 769: 
 770:         reqs_to_add: list[CachedRequestState] = []
 771:         deferred_spec_decode_corrections = []
 772: 
 773:         # Add new requests to the cached states.
 774:         for new_req_data in scheduler_output.scheduled_new_reqs:
 775:             req_id = new_req_data.req_id
 776:             if req_id in self.requests:
 777:                 # For streaming case only.
 778:                 req_state = self._update_streaming_request(req_id, new_req_data)
 779:                 reqs_to_add.append(req_state)
 780:                 continue
 781: 

In [62]:
# =============================================================================
# Step 6.4 — Inspect block-table slot mapping
# =============================================================================

print_section("Step 6.4 — Inspect block-table slot mapping")

import inspect
import vllm.v1.worker.gpu_input_batch as input_batch_module

# Find classes that define compute_slot_mapping().
for name, obj in inspect.getmembers(input_batch_module, inspect.isclass):
    if (
        hasattr(obj, "compute_slot_mapping")
        and "compute_slot_mapping" in obj.__dict__
    ):
        print("\n" + "=" * 80)
        print(name)
        print("=" * 80)
        print(inspect.getsource(obj.compute_slot_mapping))


Step 6.4 — Inspect block-table slot mapping

MultiGroupBlockTable
    def compute_slot_mapping(
        self,
        num_reqs: int,
        query_start_loc: torch.Tensor,
        positions: torch.Tensor,
    ) -> None:
        for block_table in self.block_tables:
            block_table.compute_slot_mapping(num_reqs, query_start_loc, positions)



In [63]:
# =============================================================================
# Step 6.5 — Find concrete block-table implementation
# =============================================================================

print_section("Step 6.5 — Find concrete block-table implementation")

import inspect
import vllm.v1.worker.gpu_input_batch as input_batch_module

for name, obj in inspect.getmembers(input_batch_module, inspect.isclass):
    if hasattr(obj, "compute_slot_mapping"):

        # Only show classes that define the method directly.
        if "compute_slot_mapping" in obj.__dict__:

            print("\n" + "=" * 80)
            print(name)
            print("=" * 80)

            print("Module:")
            print(obj.__module__)

            print("\nSource:")
            print(inspect.getsource(obj.compute_slot_mapping))


Step 6.5 — Find concrete block-table implementation

MultiGroupBlockTable
Module:
vllm.v1.worker.block_table

Source:
    def compute_slot_mapping(
        self,
        num_reqs: int,
        query_start_loc: torch.Tensor,
        positions: torch.Tensor,
    ) -> None:
        for block_table in self.block_tables:
            block_table.compute_slot_mapping(num_reqs, query_start_loc, positions)



In [64]:
# =============================================================================
# Step 6.5b — Find concrete per-group BlockTable implementation
# =============================================================================

print_section("Step 6.5b — Find concrete per-group BlockTable implementation")

import inspect
import vllm.v1.worker.block_table as block_table_module

for name, obj in inspect.getmembers(block_table_module, inspect.isclass):
    if (
        hasattr(obj, "compute_slot_mapping")
        and "compute_slot_mapping" in obj.__dict__
    ):
        print("\n" + "=" * 80)
        print(name)
        print("=" * 80)

        print("Module:")
        print(obj.__module__)

        print("\nSource:")
        print(inspect.getsource(obj.compute_slot_mapping))


Step 6.5b — Find concrete per-group BlockTable implementation

BlockTable
Module:
vllm.v1.worker.block_table

Source:
    def compute_slot_mapping(
        self,
        num_reqs: int,
        query_start_loc: torch.Tensor,
        positions: torch.Tensor,
    ) -> None:
        num_tokens = positions.shape[0]
        if self.slot_mapping_mode == SlotMappingMode.NONE:
            # Mamba/GDN groups consume the block table as recurrent state
            # indices and do not use per-token slot mappings.
            return
        assert self.slot_mapping_mode == SlotMappingMode.TOKEN_TO_KV_SLOT

        _COMPUTE_SLOT_MAPPING_KERNEL(
            num_reqs,
            num_tokens,
            self.max_num_batched_tokens,
            query_start_loc,
            positions,
            self.block_table.gpu,
            self.block_table.gpu.stride(0),
            self.block_size,
            self.slot_mapping.gpu,
            self.kv_cache_block_size,
            self.blocks_per_kv_block,
   

In [65]:
# =============================================================================
# Step 6.6 — Inspect slot-mapping kernel
# =============================================================================

print_section("Step 6.6 — Inspect slot-mapping kernel")

import inspect
import vllm.v1.worker.block_table as block_table_module

kernel = block_table_module._COMPUTE_SLOT_MAPPING_KERNEL

print("Kernel object:")
print(kernel)

try:
    print("\nKernel source:")
    print(inspect.getsource(kernel))
except Exception as e:
    print("\nCould not inspect source directly:")
    print(e)


Step 6.6 — Inspect slot-mapping kernel
Kernel object:

Kernel source:

Could not inspect source directly:
module, class, method, function, traceback, frame, or code object was expected, got ComputeSlotMappingKernel


In [66]:
# =============================================================================
# Step 6.6b — Inspect ComputeSlotMappingKernel class
# =============================================================================

print_section("Step 6.6b — Inspect ComputeSlotMappingKernel class")

import inspect
import vllm.v1.worker.block_table as block_table_module

KernelClass = type(block_table_module._COMPUTE_SLOT_MAPPING_KERNEL)

print("Kernel class:")
print(KernelClass)

print("\nKernel class source file:")
print(inspect.getfile(KernelClass))

print("\nKernel class source:")
print(inspect.getsource(KernelClass))


Step 6.6b — Inspect ComputeSlotMappingKernel class
Kernel class:
<class 'vllm.v1.worker.block_table.ComputeSlotMappingKernel'>

Kernel class source file:
/usr/local/lib/python3.13/dist-packages/vllm/v1/worker/block_table.py

Kernel class source:
class ComputeSlotMappingKernel(VllmJitKernel["ComputeSlotMappingKernel.CompileKey"]):
    triton_block_size = 1024

    @dataclass(frozen=True)
    class CompileKey:
        kv_cache_block_size: int
        blocks_per_kv_block: int
        total_cp_world_size: int
        total_cp_rank: int
        cp_kv_cache_interleave_size: int
        block_table_stride: int
        block_size: int

    @staticmethod
    @triton.jit(do_not_specialize=["num_tokens", "max_num_tokens"])
    def kernel(
        num_tokens,
        max_num_tokens,
        query_start_loc_ptr,  # [num_reqs + 1], int32
        positions_ptr,  # [num_tokens], int64
        block_table_ptr,  # [max_num_reqs, max_num_blocks_per_req], int32 (flat)
        block_table_stride,  # max_n